# ASR Whisper Training on Google Colab A100
Run with Runtime → GPU A100.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi


## Bootstrap (auto-detect Drive path)
`USE_LOCAL_SSD=1` creates a temporary Colab runtime copy for speed, not a permanent Drive duplicate.

This notebook now auto-detects `Colab_ASR_A100_Training` so it works even if the package is not exactly under `MyDrive/ASR_Colab_A100/`.


In [ ]:
import os, subprocess, pathlib, textwrap

os.environ['USE_LOCAL_SSD'] = '1'  # copy Drive dataset + split TSV to /content SSD
os.environ.setdefault('MIN_LOCAL_FREE_GB', '40')
os.environ.setdefault('USE_DATA_ARCHIVE', '1')  # use Data/_archives/*.tar if present (fastest bootstrap)
os.environ.setdefault('A100_SYNC_INTERVAL_SEC', '600')
# Set to '1' only if you want Colab runtime to auto-disconnect after final sync + summary.
os.environ.setdefault('A100_AUTO_DISCONNECT', '0')

# Fast common paths first, then bounded find. Edit MANUAL_COLAB_ROOT if auto-detect fails.
MANUAL_COLAB_ROOT = ''  # e.g. '/content/drive/MyDrive/ASR_Colab_A100/Colab_ASR_A100_Training'
common_candidates = [
    MANUAL_COLAB_ROOT,
    '/content/drive/MyDrive/ASR_Colab_A100/Colab_ASR_A100_Training',
    '/content/drive/MyDrive/Colab_ASR_A100_Training',
]
found = []
for c in common_candidates:
    if c and pathlib.Path(c, 'scripts', 'colab_bootstrap_a100.sh').exists():
        found.append(c)

if not found:
    cmd = "find /content/drive/MyDrive /content/drive/Shareddrives -maxdepth 6 -type f -path '*/Colab_ASR_A100_Training/scripts/colab_bootstrap_a100.sh' 2>/dev/null | head -20"
    out = subprocess.getoutput(cmd).strip().splitlines()
    found = [str(pathlib.Path(x).parents[1]) for x in out if x.strip()]

if not found:
    raise FileNotFoundError('Cannot find Colab_ASR_A100_Training/scripts/colab_bootstrap_a100.sh in Google Drive. Upload the whole Colab_ASR_A100_Training folder or set MANUAL_COLAB_ROOT to its exact path.')

DRIVE_COLAB_ROOT = found[0]
DRIVE_PROJECT_ROOT = str(pathlib.Path(DRIVE_COLAB_ROOT).parent)
DRIVE_RESULTS_ROOT = str(pathlib.Path(DRIVE_PROJECT_ROOT) / 'Results')

os.environ['DRIVE_COLAB_ROOT'] = DRIVE_COLAB_ROOT
os.environ['DRIVE_PROJECT_ROOT'] = DRIVE_PROJECT_ROOT
os.environ['DRIVE_RESULTS_ROOT'] = DRIVE_RESULTS_ROOT

print('DRIVE_PROJECT_ROOT =', DRIVE_PROJECT_ROOT)
print('DRIVE_COLAB_ROOT   =', DRIVE_COLAB_ROOT)
print('DRIVE_RESULTS_ROOT =', DRIVE_RESULTS_ROOT)
print('USE_LOCAL_SSD      =', os.environ['USE_LOCAL_SSD'])
print('MIN_LOCAL_FREE_GB   =', os.environ['MIN_LOCAL_FREE_GB'])
print('USE_DATA_ARCHIVE   =', os.environ['USE_DATA_ARCHIVE'])
print('A100_SYNC_INTERVAL_SEC =', os.environ['A100_SYNC_INTERVAL_SEC'])
print('A100_AUTO_DISCONNECT =', os.environ['A100_AUTO_DISCONNECT'])

!bash "$DRIVE_COLAB_ROOT/scripts/colab_bootstrap_a100.sh"


## Train Whisper-small — paper-exact profile (recommended for paper)
Effective batch 32: batch 8 x grad_accum 4. This matches `RUN_GUIDE.md` and is the most defensible paper run.


In [ ]:
!bash "$DRIVE_COLAB_ROOT/scripts/colab_train_m02b_whisper_small_paper_exact.sh"


## Optional A100-fast Whisper-small
Use only if you explicitly choose speed over paper-exact microbatch parity. Effective batch remains 32, but microbatch/checkpointing differ.


In [ ]:
# !bash "$DRIVE_COLAB_ROOT/scripts/colab_train_m02b_whisper_small_a100_fast.sh"


## Optional Whisper-medium on A100


In [ ]:
# !bash "$DRIVE_COLAB_ROOT/scripts/colab_train_m02b_whisper_medium_a100.sh"


## Paper-ready training time and metric summary
Run this after training/test finishes. It reads `log.txt`, `report.md`, and `test_results/test_paper.json`, then writes a summary into Drive `Results/paper_training_time_summary.md`.


In [ ]:
import os, json, re, datetime
from pathlib import Path

results_root = Path(os.environ.get('DRIVE_RESULTS_ROOT', '/content/drive/MyDrive/ASR_Colab_A100/Results'))
print('Results root:', results_root)

run_dirs = []
for family in ['m02b_whisper_small_ft', 'm02b_whisper_medium_ft']:
    base = results_root / family
    if base.exists():
        run_dirs.extend([p for p in base.glob('run_paper_*') if p.is_dir()])
run_dirs = sorted(run_dirs, key=lambda p: p.stat().st_mtime, reverse=True)

rows = []
for run in run_dirs:
    log_path = run / 'log.txt'
    report_path = run / 'report.md'
    test_json = run / 'test_results' / 'test_paper.json'
    total_time = 'MISSING'
    hhmmss = ''
    if log_path.exists():
        m = re.findall(r'Total waktu training:\s*(.+)', log_path.read_text(encoding='utf-8', errors='ignore'))
        if m:
            total_time = m[-1].strip()
    if report_path.exists():
        m = re.findall(r'Total training time:\s*([^\n]+)', report_path.read_text(encoding='utf-8', errors='ignore'))
        if m:
            hhmmss = m[-1].strip()
    metrics = {}
    model_id = run.parent.name
    if test_json.exists():
        data = json.loads(test_json.read_text(encoding='utf-8'))
        model_id = data.get('model_id', model_id)
        metrics = data.get('metrics', {})
    rows.append({
        'run': run.name,
        'model_id': model_id,
        'total_waktu_training': total_time,
        'total_training_time_report': hhmmss,
        'wer': metrics.get('wer', 'MISSING'),
        'cer': metrics.get('cer', 'MISSING'),
        'test_json': str(test_json) if test_json.exists() else 'MISSING',
        'run_dir': str(run),
    })

if not rows:
    print('No Colab Whisper result runs found yet.')
else:
    print('\nLatest Colab Whisper runs:')
    for r in rows[:10]:
        print(f"- {r['model_id']} | {r['run']} | total={r['total_waktu_training']} | WER={r['wer']} | CER={r['cer']}")

    out_md = results_root / 'paper_training_time_summary.md'
    out_json = results_root / 'paper_training_time_summary.json'
    lines = []
    lines.append('# Paper Training Time Summary - Colab Whisper\n\n')
    lines.append(f'Generated: {datetime.datetime.now().isoformat()}\n\n')
    lines.append('| model_id | run | total_waktu_training | report_time | WER | CER | test_json |\n')
    lines.append('|---|---|---:|---:|---:|---:|---|\n')
    for r in rows:
        lines.append(f"| {r['model_id']} | {r['run']} | {r['total_waktu_training']} | {r['total_training_time_report']} | {r['wer']} | {r['cer']} | `{r['test_json']}` |\n")
    out_md.write_text(''.join(lines), encoding='utf-8')
    out_json.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding='utf-8')
    print('\nWrote:', out_md)
    print('Wrote:', out_json)
